In [1]:
import os
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm
 
from scipy.stats import norm
from scipy.optimize import minimize
from scipy.interpolate import interp1d

In [2]:
# ----------------------------- config ---------------------------------------
EPS = 1e-6
MIN_SAMPLES = 10
N_DRESS = 50
N_QUANTILES = 200
RNG_SEED = 42
 
q_levels = np.linspace(0.0025, 0.9975, N_QUANTILES)
 
years_train = np.arange(1959, 2000)
years_verif = np.arange(2000, 2021)
T_CENTER = float(years_train.mean())
 # MDD
vars_keep = [
 'Fort_Bragg_PRECT_1d_max',
 'Fort_Bragg_PRECT_3d_max',
 'Fort_Bragg_PRECT_5d_max',
 'Fort_Bragg_simple_PRECT_30d',
 'Fort_Bragg_simple_PRECT_max',
 'Fort_Bragg_simple_PRECT_mean',
 'Guam_simple_PRECT_30d',
 'Guam_simple_PRECT_max',
 'Guam_simple_PRECT_mean',
 'Pituffik_FT_days',
 'Pituffik_MDD',
 'Pituffik_simple_PRECT_30d',
 'Pituffik_simple_PRECT_max',
 'Pituffik_simple_PRECT_mean',
 'Yuma_PG_PRECT_1d_max',
 'Yuma_PG_PRECT_7d_max',
 'Yuma_PG_simple_PRECT_30d',
 'Yuma_PG_simple_PRECT_max',
 'Yuma_PG_simple_PRECT_mean',
]
 
data_dir = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/'
in_input  = os.path.join(data_dir, 'STN_CESM_ALL_20260604.zarr')
in_target = os.path.join(data_dir, 'STN_ERA5_ALL_20260604.zarr')

# ----------------------------- load -----------------------------------------
ds_input = xr.open_zarr(in_input)[vars_keep].drop_vars('valid_year', errors='ignore')
ds_input_train = ds_input.sel(init_year=years_train)
ds_input_verif = ds_input.sel(init_year=years_verif)
 
ds_target = xr.open_zarr(in_target)[vars_keep]
 
n_member = ds_input_train.sizes['member']
n_lead   = ds_input_train.sizes['lead_year']
n_train  = ds_input_train.sizes['init_year']
n_verif  = ds_input_verif.sizes['init_year']
 
lead_years        = ds_input_train['lead_year'].values
init_years_train  = ds_input_train['init_year'].values
init_years_verif  = ds_input_verif['init_year'].values

In [3]:
# ============================================================================
# Normal quantile transform (user-supplied, verbatim)
# ============================================================================
def build_quantile_table(samples):
    """Build empirical quantile table from 1D samples."""
    valid = samples[~np.isnan(samples)]
    if len(valid) < MIN_SAMPLES:
        return np.full(N_QUANTILES, np.nan), np.nan
    q_values = np.quantile(valid, q_levels).astype(np.float32)
    wet_frac = np.float32(np.mean(valid > 0.0))
    return q_values, wet_frac
 
 
def extrapolate_upper_tail(q_vals, q_levs):
    """Linearly extrapolate to quantile level 1.0."""
    v1, v2 = q_vals[-2], q_vals[-1]
    l1, l2 = q_levs[-2], q_levs[-1]
    slope = (v2 - v1) / max(l2 - l1, EPS)
    v_max = max(v2 + slope * (1.0 - l2), v2)
    return np.append(q_vals, np.float32(v_max)), np.append(q_levs, 1.0)
 
 
def precip_to_gaussian(values, q_values, wet_frac, rng=None):
    """Map precipitation -> Gaussian via NQT."""
    n = len(values)
    gaussian = np.full(n, np.nan, dtype=np.float32)
 
    if np.all(np.isnan(q_values)):
        return gaussian
    if rng is None:
        rng = np.random.default_rng()
 
    valid = ~np.isnan(values)
    vals = np.maximum(values[valid], 0.0)
 
    dry_frac = 1.0 - wet_frac
    u = np.full(len(vals), np.nan, dtype=np.float64)
 
    is_zero = vals <= 0.0
    is_wet = ~is_zero
 
    if dry_frac > 0 and is_zero.any():
        u[is_zero] = rng.uniform(EPS, max(dry_frac, 2 * EPS), size=is_zero.sum())
 
    if is_wet.any():
        wet_q_mask = q_values > 0.0
        if wet_q_mask.sum() >= 2:
            q_wet = q_values[wet_q_mask]
            l_wet = q_levels[wet_q_mask]
            q_ext, l_ext = extrapolate_upper_tail(q_wet, l_wet)
 
            q_unique, idx_unique = np.unique(q_ext, return_index=True)
            ql_unique = l_ext[idx_unique]
 
            if len(q_unique) >= 2:
                interp_func = interp1d(
                    q_unique, ql_unique,
                    kind='linear', bounds_error=False,
                    fill_value=(ql_unique[0], ql_unique[-1]),
                )
                u_wet_raw = interp_func(vals[is_wet])
                l_min, l_max = ql_unique[0], ql_unique[-1]
                u[is_wet] = dry_frac + (1.0 - dry_frac) * (
                    (u_wet_raw - l_min) / max(l_max - l_min, EPS)
                )
            else:
                u[is_wet] = dry_frac + 0.5 * (1.0 - dry_frac)
        else:
            u[is_wet] = dry_frac + 0.5 * (1.0 - dry_frac)
 
    u = np.clip(u, 1e-6, 1.0 - 1e-6)
    gaussian[valid] = norm.ppf(u).astype(np.float32)
    return gaussian
 
 
def gaussian_to_precip(gaussian_values, q_values, wet_frac):
    """Inverse NQT: Gaussian -> precipitation (mm/day)."""
    shape = gaussian_values.shape
    g_flat = gaussian_values.ravel()
    precip = np.full_like(g_flat, np.nan, dtype=np.float32)
 
    if np.all(np.isnan(q_values)):
        return precip.reshape(shape)
 
    valid = ~np.isnan(g_flat)
    if not valid.any():
        return precip.reshape(shape)
 
    u = norm.cdf(g_flat[valid])
    dry_frac = 1.0 - wet_frac
    result = np.zeros_like(u, dtype=np.float32)
 
    wet_mask = u > dry_frac
    if wet_mask.any():
        wet_q_mask = q_values > 0.0
        if wet_q_mask.sum() >= 2:
            q_wet = q_values[wet_q_mask]
            l_wet = q_levels[wet_q_mask]
            q_ext, l_ext = extrapolate_upper_tail(q_wet, l_wet)
 
            q_unique, idx_unique = np.unique(q_ext, return_index=True)
            ql_unique = l_ext[idx_unique]
 
            if len(q_unique) >= 2:
                l_min, l_max = ql_unique[0], ql_unique[-1]
                u_rescaled = l_min + (l_max - l_min) * (
                    (u[wet_mask] - dry_frac) / max(1.0 - dry_frac, EPS)
                )
                inv_func = interp1d(
                    ql_unique, q_unique,
                    kind='linear', bounds_error=False,
                    fill_value=(q_unique[0], q_unique[-1]),
                )
                result[wet_mask] = np.maximum(inv_func(u_rescaled), 0.0)
            else:
                result[wet_mask] = q_values[wet_q_mask].mean()
        elif wet_q_mask.any():
            result[wet_mask] = q_values[wet_q_mask][0]
 
    precip[valid] = result.astype(np.float32)
    return precip.reshape(shape)

In [4]:
# ============================================================================
# EMOS helpers
# ============================================================================
def crps_gaussian(mu, sigma, y):
    sigma = np.maximum(sigma, EPS)
    z = (y - mu) / sigma
    return sigma * (z * (2.0 * norm.cdf(z) - 1.0)
                    + 2.0 * norm.pdf(z)
                    - 1.0 / np.sqrt(np.pi))
 
def fit_emos_standardized(xbar, sx2, y, gamma_min=0.0):
    valid = np.isfinite(xbar) & np.isfinite(sx2) & np.isfinite(y)
    if valid.sum() < MIN_SAMPLES:
        return np.full(4, np.nan), False
    xbar_v, sx2_v, y_v = xbar[valid], sx2[valid], y[valid]

    # Training-period scalings
    xbar_mean = float(np.mean(xbar_v))
    xbar_std  = float(np.std(xbar_v, ddof=1))
    y_mean    = float(np.mean(y_v))
    y_std     = float(np.std(y_v, ddof=1))

    # Standardized predictor and target
    xs = (xbar_v - xbar_mean) / max(xbar_std, EPS)
    ys = (y_v    - y_mean)    / max(y_std,    EPS)
    sx2_norm = sx2_v / max(xbar_std ** 2, EPS)

    # OLS init: gamma = corr(xs, ys), tau in [0, 1]-ish for spread coupling
    gamma_0 = float(np.clip(np.corrcoef(xs, ys)[0, 1], gamma_min, 1.0))
    resid2 = (ys - gamma_0 * xs) ** 2
    try:
        B = np.column_stack([np.ones_like(sx2_norm), sx2_norm])
        cd2, *_ = np.linalg.lstsq(B, resid2, rcond=None)
        c2_0 = max(float(cd2[0]), EPS)
        d2_0 = max(float(cd2[1]), EPS)
    except Exception:
        c2_0, d2_0 = float(np.var(ys)) + EPS, 0.0

    def obj(p):
        gamma, c, d = p
        mu_s = gamma * xs
        sigma_s = np.sqrt(np.maximum(c * c + d * d * sx2_norm, EPS * EPS))
        return float(np.mean(crps_gaussian(mu_s, sigma_s, ys)))

    x0 = np.array([gamma_0, np.sqrt(c2_0), np.sqrt(d2_0)])
    bounds = [(gamma_min, 1.0), (None, None), (None, None)]
    res = minimize(obj, x0, method='L-BFGS-B', bounds=bounds)
    gamma_hat, c_hat, d_hat = res.x

    # Convert back to (a, b, c, d) in original units for downstream compatibility
    b_hat = gamma_hat * y_std / max(xbar_std, EPS)
    a_hat = y_mean - b_hat * xbar_mean
    # Spread coefficients: c, d operate on standardized variance, scale back to y units
    c_phys = c_hat * y_std
    d_phys = d_hat * y_std / max(xbar_std, EPS)

    return np.array([a_hat, b_hat, c_phys, d_phys]), bool(res.success)
 
 
def lookup_target(y_full, valid_years_needed, valid_years_avail):
    out = np.full(len(valid_years_needed), np.nan, dtype=np.float64)
    in_mask = np.isin(valid_years_needed, valid_years_avail)
    if in_mask.any():
        out[in_mask] = y_full.sel(valid_year=valid_years_needed[in_mask]).values
    return out

In [5]:
# ============================================================================
# 1. Build training-period NQT tables per (variable, lead_year)
# ============================================================================
rng = np.random.default_rng(RNG_SEED)
 
qt_smyle = {}    # (var, il) -> (q_values, wet_frac)
qt_era5  = {}
 
for var in tqdm(vars_keep, desc='Build NQT tables'):
    x_train = ds_input_train[var].values          # (n_train, n_lead, n_member)
    y_full  = ds_target[var]
    valid_years_avail = y_full['valid_year'].values
 
    for il, lead in enumerate(lead_years):
        smyle_sample = x_train[:, il, :].ravel()
        qt_smyle[(var, il)] = build_quantile_table(smyle_sample)
 
        valid_yrs_needed = init_years_train + int(lead)
        era5_sample = lookup_target(y_full, valid_yrs_needed, valid_years_avail)
        qt_era5[(var, il)] = build_quantile_table(era5_sample)

Build NQT tables: 100%|██████████| 19/19 [00:31<00:00,  1.67s/it]


In [11]:
# ============================================================================
# 2. Gaussianize training data and fit EMOS in z-space
# ============================================================================
params_dict = {v: np.full((n_lead, 4), np.nan) for v in vars_keep}
fit_flags   = {v: np.zeros(n_lead, dtype=bool) for v in vars_keep}
 
for var in tqdm(vars_keep, desc='EMOS training   '):
    x_train = ds_input_train[var].values
    y_full  = ds_target[var]
    valid_years_avail = y_full['valid_year'].values
 
    for il, lead in enumerate(lead_years):
        q_s, w_s = qt_smyle[(var, il)]
        q_e, w_e = qt_era5[(var, il)]
        if np.all(np.isnan(q_s)) or np.all(np.isnan(q_e)):
            continue
 
        # Per-member NQT: flatten -> transform -> reshape preserves spread
        x_il = x_train[:, il, :]                                  # (n_train, n_member)
        z_smyle = precip_to_gaussian(x_il.ravel().astype(np.float64),
                                     q_s, w_s, rng).reshape(x_il.shape)
        xbar = np.nanmean(z_smyle, axis=1)
        sx2  = np.nanvar (z_smyle, axis=1, ddof=1)
 
        valid_yrs_needed = init_years_train + int(lead)
        y_phys = lookup_target(y_full, valid_yrs_needed, valid_years_avail)
        z_era5 = precip_to_gaussian(y_phys.astype(np.float64), q_e, w_e, rng)
 
        params, success = fit_emos_standardized(xbar, sx2, z_era5, gamma_min=0.1)
        params_dict[var][il, :] = params
        fit_flags[var][il]      = success

EMOS training   : 100%|██████████| 19/19 [00:22<00:00,  1.18s/it]


In [13]:
# ============================================================================
# 3. Apply calibration on verification period
# 4. Dress 50 members in z-space, back-transform through ERA5 NQT^-1
# ============================================================================
calib_data = {}
dress_data = {}
 
for var in tqdm(vars_keep, desc='EMOS apply      '):
    x_verif = ds_input_verif[var].values
    P = params_dict[var]
 
    mu_arr     = np.full((n_verif, n_lead), np.nan)
    sigma_arr  = np.full((n_verif, n_lead), np.nan)
    dress_phys = np.full((n_verif, n_lead, N_DRESS), np.nan)
 
    for il in range(n_lead):
        a, b, c, d = P[il]
        if not np.isfinite(a):
            continue
        q_s, w_s = qt_smyle[(var, il)]
        q_e, w_e = qt_era5[(var, il)]
        if np.all(np.isnan(q_s)) or np.all(np.isnan(q_e)):
            continue
 
        x_il = x_verif[:, il, :]
        z_smyle = precip_to_gaussian(x_il.ravel().astype(np.float64),
                                     q_s, w_s, rng).reshape(x_il.shape)
        xbar = np.nanmean(z_smyle, axis=1)
        sx2  = np.nanvar (z_smyle, axis=1, ddof=1)
 
        mu = a + b * xbar
        sigma = np.sqrt(np.maximum(c * c + d * d * sx2, EPS * EPS))
        mu_arr[:, il]    = mu
        sigma_arr[:, il] = sigma
 
        z_draw = mu[:, None] + sigma[:, None] * rng.standard_normal((n_verif, N_DRESS))
        dress_phys[:, il, :] = gaussian_to_precip(z_draw.astype(np.float32),
                                                  q_e, w_e)
 
    calib_data[var + '_mu_z']    = (('init_year', 'lead_year'), mu_arr)
    calib_data[var + '_sigma_z'] = (('init_year', 'lead_year'), sigma_arr)
    dress_data[var] = (('init_year', 'lead_year', 'member'), dress_phys)
 
ds_calib = xr.Dataset(
    calib_data,
    coords={'init_year': init_years_verif, 'lead_year': lead_years},
)
ds_dress = xr.Dataset(
    dress_data,
    coords={
        'init_year': init_years_verif,
        'lead_year': lead_years,
        'member':    np.arange(N_DRESS),
    },
)
 
param_arr = np.stack([params_dict[v] for v in vars_keep], axis=0)
ds_params = xr.Dataset(
    {'emos_params': (('variable', 'lead_year', 'param'), param_arr),
     'fit_success': (('variable', 'lead_year'),
                     np.stack([fit_flags[v] for v in vars_keep], axis=0))},
    coords={
        'variable':  vars_keep,
        'lead_year': lead_years,
        'param':     ['a', 'b', 'c', 'd'],
    },
)

EMOS apply      : 100%|██████████| 19/19 [00:08<00:00,  2.21it/s]


In [14]:
# ============================================================================
# 5. Verification (physical units)
# ============================================================================
def crps_ensemble(ens, y):
    out = np.full(ens.shape[0], np.nan)
    finite_ens = np.all(np.isfinite(ens), axis=1)
    mask = finite_ens & np.isfinite(y)
    if not mask.any():
        return out
    e = ens[mask]; yv = y[mask]
    term1 = np.mean(np.abs(e - yv[:, None]), axis=1)
    diff  = np.abs(e[:, :, None] - e[:, None, :])
    term2 = 0.5 * diff.mean(axis=(1, 2))
    out[mask] = term1 - term2
    return out
 
 
def anomaly_correlation(fc, obs):
    m = np.isfinite(fc) & np.isfinite(obs)
    if m.sum() < 2:
        return np.nan
    fa = fc[m]  - np.mean(fc[m])
    oa = obs[m] - np.mean(obs[m])
    den = np.sqrt(np.sum(fa * fa) * np.sum(oa * oa))
    return float(np.sum(fa * oa) / den) if den > 0 else np.nan
 
 
verif_rows = []
for var in tqdm(vars_keep, desc='Verification    '):
    y_full = ds_target[var]
    valid_years_avail = y_full['valid_year'].values
 
    x_raw   = ds_input_verif[var].values
    x_dress = ds_dress[var].values
 
    for il, lead in enumerate(lead_years):
        valid_yrs = init_years_verif + int(lead)
        y_arr = lookup_target(y_full, valid_yrs, valid_years_avail)
 
        crps_raw   = crps_ensemble(x_raw[:, il, :],   y_arr)
        crps_dress = crps_ensemble(x_dress[:, il, :], y_arr)
 
        raw_mean   = np.nanmean(x_raw[:, il, :],   axis=1)
        dress_mean = np.nanmean(x_dress[:, il, :], axis=1)
 
        acc_raw   = anomaly_correlation(raw_mean,   y_arr)
        acc_dress = anomaly_correlation(dress_mean, y_arr)
 
        mean_raw   = float(np.nanmean(crps_raw))
        mean_dress = float(np.nanmean(crps_dress))
        crpss = (1.0 - mean_dress / mean_raw) if (mean_raw and mean_raw > 0) else np.nan
 
        # Diagnostic: dry_frac in the ERA5 table at this lead
        _, w_e = qt_era5[(var, il)]
        dry_frac_era5 = float(1.0 - w_e) if np.isfinite(w_e) else np.nan
 
        verif_rows.append({
            'variable':       var,
            'lead_year':      int(lead),
            'n_eff':          int(np.isfinite(y_arr).sum()),
            'dry_frac_era5':  dry_frac_era5,
            'CRPS_raw':       mean_raw,
            'CRPS_dress':     mean_dress,
            'CRPSS':          crpss,
            'ACC_raw':        acc_raw,
            'ACC_dress':      acc_dress,
            'b':              float(params_dict[var][il, 1]),
            'fit_ok':         bool(fit_flags[var][il]),
        })
 
df_verif = pd.DataFrame(verif_rows)

Verification    : 100%|██████████| 19/19 [00:10<00:00,  1.85it/s]


In [16]:
save_dir = '/glade/derecho/scratch/ksha/EPRI_data/PP_calib/EMOS/'
ds_dress.to_zarr(save_dir+'Dress_QM.zarr', mode='w')
ds_calib.to_zarr(save_dir+'Calib_QM.zarr', mode='w')

In [15]:
print("\n=== EMOS with trend ===")
summary = df_verif.groupby('variable')[['CRPS_raw', 'CRPS_dress', 'CRPSS', 'ACC_raw', 'ACC_dress']].mean().round(3)
print(summary.to_string())


=== EMOS with trend ===
                              CRPS_raw  CRPS_dress  CRPSS  ACC_raw  ACC_dress
variable                                                                     
Fort_Bragg_PRECT_1d_max         21.051      19.446  0.074    0.090      0.190
Fort_Bragg_PRECT_3d_max         10.634      10.427  0.020    0.080      0.030
Fort_Bragg_PRECT_5d_max          6.852       6.769  0.012    0.108     -0.057
Fort_Bragg_simple_PRECT_30d      1.366       1.359  0.005   -0.030     -0.155
Fort_Bragg_simple_PRECT_max     21.051      20.333  0.034    0.090     -0.083
Fort_Bragg_simple_PRECT_mean     0.533       0.327  0.376    0.084     -0.018
Guam_simple_PRECT_30d            2.489       2.553 -0.037    0.134      0.190
Guam_simple_PRECT_max           26.873      28.003 -0.045    0.106      0.095
Guam_simple_PRECT_mean           0.697       0.736 -0.070    0.073      0.041
Pituffik_FT_days                 3.731       3.811 -0.023    0.016     -0.038
Pituffik_MDD                    64.450 